# Module 7 project-exercise demo — Balancing, co-activation, and interventions

**Implementation demo, not a completed full-scale project.** Synthetic clusters make routes observable. Router and expert weights are fixed; the clusters are not evidence of semantic specialization.
All inputs are small, deterministic, offline, and CPU-friendly. The goal is to
validate interfaces, metrics, invariants, and report structure before spending
GPU or API budget.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"reproducible smoke tier; seed={SEED}")
DEMO_LANGUAGE = 'en'

plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":12, "axes.titlesize":15, "axes.labelsize":13, "xtick.labelsize":11, "ytick.labelsize":11, "figure.dpi":150})


## Demo 1. Balancing without training the router

Generate 900 vectors in three synthetic clusters and route each to two of six experts. Freeze the router weights and update only the selection biases for 100 steps, using the sign of each expert's deviation from average load. This isolates the controller from weight learning.

The left plot tracks load variation. The right plot divides each cluster column by 600: 300 tokens times two assignments. Columns therefore sum to one and represent assignment shares, rather than the fraction of tokens that selected an expert.

In [ ]:
ru = DEMO_LANGUAGE == "ru"
rng = np.random.default_rng(SEED)
cluster = np.repeat(np.arange(3), 300)
features = rng.normal(size=(900, 8)) + np.eye(3, 8)[cluster] * 2
router_w = rng.normal(size=(8, 6))
logits = features @ router_w
log_scores = -np.logaddexp(0., -logits)
scores = np.exp(log_scores)

def route(bias):
    indices = np.argsort(-(scores + bias), axis=1, kind="stable")[:, :2]
    selected = np.take_along_axis(log_scores, indices, axis=1)
    gates = np.exp(selected - selected.max(axis=1, keepdims=True))
    gates /= gates.sum(axis=1, keepdims=True)
    return indices, gates, np.bincount(indices.ravel(), minlength=6)

bias = np.zeros(6)
history = []
for step in range(101):
    top2, gates, counts = route(bias)
    history.append(counts.std() / counts.mean())
    if step < 100:
        bias += .01 * np.sign(counts.mean() - counts)
assert history[-1] < history[0]
assert counts.sum() == 1800
print("CV:", round(history[0], 4), "→", round(history[-1], 4))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), layout="constrained")
axes[0].plot(history, color="#5E81AC")
axes[0].set(xlabel="Обновления смещений" if ru else "Bias updates", ylabel="CV",
            title="Фиксированные оценки" if ru else "Fixed router scores")
heat = np.zeros((6, 3), dtype=int)
for c, selected in zip(cluster, top2):
    heat[selected, c] += 1
im = axes[1].imshow(heat / 600, cmap="Blues", vmin=0, vmax=1, aspect="auto")
axes[1].set(xlabel="Синтетический кластер" if ru else "Synthetic cluster",
            ylabel="Эксперт" if ru else "Expert", xticks=[0,1,2],
            title="Доля назначений в кластере" if ru else "Assignment share within cluster")
fig.colorbar(im, ax=axes[1])
assert np.allclose((heat / 600).sum(axis=0), 1.)
plt.show()

Lower CV establishes better balance for this fixed score population. Cluster columns may still differ because aggregate balance does not require identical routes in every cluster. The experiment measures neither task quality nor semantic expertise.

## Demo 2. Co-selection versus intervention effects

Each top-2 pair contributes two directed counts. Dividing a row by that expert's selection count gives the conditional distribution of its partner. Next create fixed random FFNs and zero each expert's contribution in turn. Keep routes and all remaining gates fixed; renormalizing or rerouting would define a different intervention.

In [ ]:
# Count co-activation separately from the effect of changing expert outputs.
co = np.zeros((6, 6), dtype=int)
for a, b in top2:
    co[a, b] += 1
    co[b, a] += 1
assert np.all(co == co.T) and np.all(np.diag(co) == 0)
assert np.array_equal(co.sum(axis=1), counts)  # exactly one partner at top-2
conditional = co / counts[:, None]
assert np.allclose(conditional.sum(axis=1), 1.)
w1 = rng.normal(scale=.3, size=(6, 8, 12))
w2 = rng.normal(scale=.3, size=(6, 12, 8))
expert_outputs = np.stack([np.tanh(features @ a) @ b for a,b in zip(w1,w2)], axis=1)
selected_outputs = expert_outputs[np.arange(len(features))[:, None], top2]
original = (gates[..., None] * selected_outputs).sum(axis=1)
effects = []
for expert in range(6):
    # Zero an expert contribution while keeping routes and all other gates fixed.
    kept = gates * (top2 != expert)
    intervened = (kept[..., None] * selected_outputs).sum(axis=1)
    effects.append(np.mean((intervened - original)**2))
assert np.all(np.isfinite(effects)) and min(effects) >= 0
fig, axes = plt.subplots(1,2,figsize=(11,4.8),layout="constrained")
im=axes[0].imshow(conditional,cmap="Blues",vmin=0,vmax=1)
axes[0].set(xlabel="Партнёр" if ru else "Partner expert",ylabel="Выбранный эксперт" if ru else "Selected expert",
    title="P(партнёр | эксперт)" if ru else "P(partner | expert)")
fig.colorbar(im,ax=axes[0])
axes[1].bar(range(6),effects,color="#D08770")
axes[1].set(xlabel="Обнулённый эксперт" if ru else "Zeroed expert",
    ylabel="MSE выхода" if ru else "Output MSE",
    title="При неизменных маршрутах" if ru else "With routes held fixed")
plt.show()
print("Это изменение вектора, не падение качества задачи." if ru else "This measures vector change, not task-quality loss.")

Raw co-selection counts are symmetric, whereas conditional partner probabilities need not be: their row denominators differ. The bars measure mean squared output-vector changes. Claims about a lost capability would require a trained model and a task-level evaluation.

## Scaling to the full project

Use held-out documents for a trained model. Compare frequencies, conditional co-activation, and task-metric changes under a declared intervention policy. Distinguish zeroing a contribution from renormalizing gates or rerouting.

### Minimum report

1. Data, model, and code revisions, plus the random seed.
2. The primary metric, plus quality, cost, latency, and memory where applicable.
3. A baseline, a matched budget, and at least one negative control.
4. Failed runs and the limits of the supported conclusions.